# 04. Credit Risk Scorecard Development & Risk-Band Allocation
## Financial Analytics and Risk Reporting System

**Objective:** Train an interpretable logistic regression credit scorecard on Weight of Evidence features, scale log-odds to standard scorecard points (Offset + Factor * ln(odds)), assign borrowers into 4 risk bands, and compute portfolio Expected Loss (PD * LGD * EAD).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")
from src.config import get_data_path
from src.credit_scorecard import CreditScorecard

print("Credit scorecard setup complete.")


### 1. Load Feature Engineered Loan Dataset


In [ ]:
stg_dir = get_data_path("staging")
df_loans = pd.read_csv(os.path.join(stg_dir, "stg_loans.csv"))
df_model = df_loans.dropna(subset=["default_flag"]).copy()

features = ["dti", "annual_inc", "loan_amnt", "int_rate", "revol_util", "inq_last_6mths"]
X = df_model[features].fillna(df_model[features].median())
y = df_model["default_flag"].astype(int)

print(f"Training set: {X.shape[0]} samples, {X.shape[1]} features. Event rate: {y.mean()*100:.2f}%")


### 2. Fit Credit Scorecard Model & Scale to Points


In [ ]:
scorecard = CreditScorecard(base_score=600, base_odds=2.0, pdo=30, lgd_assumption=0.45)
scorecard.fit(X, y, feature_names=features)

eval_results = scorecard.evaluate(X, y)
print("Model Evaluation Metrics:")
for k, v in eval_results.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v}")


### 3. ROC Curve and Discrimination Validation


In [ ]:
fig = scorecard.plot_roc_curve(X, y)
plt.show()


### 4. Score Distribution and Risk Band Allocation


In [ ]:
scores = scorecard.calculate_score_from_pd(scorecard.predict_proba(X))
fig = scorecard.plot_score_distribution(scores, y_true=y.values)
plt.show()


### 5. Portfolio Expected Loss Estimation


In [ ]:
pd_full = scorecard.predict_proba(df_loans[features].fillna(df_loans[features].median()))
scores_full = scorecard.calculate_score_from_pd(pd_full)
bands_full = scorecard.assign_risk_band(scores_full)
el_full = scorecard.calculate_expected_loss(pd_full, ead=df_loans["loan_amnt"].values)

df_loans["scorecard_points"] = scores_full
df_loans["risk_band"] = bands_full
df_loans["expected_loss"] = el_full

band_summary = df_loans.groupby("risk_band").agg(
    loan_count=("loan_amnt", "count"),
    total_ead=("loan_amnt", "sum"),
    total_expected_loss=("expected_loss", "sum"),
    avg_score=("scorecard_points", "mean")
).reset_index()
print("Risk Band Portfolio Summary:")
print(band_summary)
